# 第 5 章 · 马尔可夫决策过程 MDP：环境的"说明书" p(s', r | s, a)

本章回答：环境怎么回应智能体的动作？能不能用一张表把环境完全写下来？"马尔可夫"是什么意思？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/05-mdp.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 那张表 | `P[s][a] = [(概率, 下一个状态, 奖励, 是否结束), ...]` | 打印 5 组 |
| 每组加起来是 1 | `check_sums(P)` | 全是 1.0 |
| $p(\text{高电} \mid \text{低电}, \text{搜索})$ | `transition_prob(P, 1, 0, 0)` | 0.30000000000000004 |
| $r(\text{低电}, \text{搜索})$ | `expected_reward(P, 1, 0)` | 0.5 |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `ROBOT_ACTIONS`**（来自 `rl_envs.py`）

In [ ]:
ROBOT_ACTIONS = ["搜索", "等待", "充电"]

**常量 `ROBOT_STATES`**（来自 `rl_envs.py`）

In [ ]:
ROBOT_STATES = ["高电", "低电"]

**函数 `check_sums`**（来自 `ch05_mdp.py`）：下面要用到的小工具。

In [ ]:
def check_sums(P):
    return {(s, a): sum(p for p, *_ in P[s][a]) for s in P for a in P[s]}

**函数 `expected_reward`**（来自 `ch05_mdp.py`）：下面要用到的小工具。

In [ ]:
def expected_reward(P, s, a):
    return sum(p * r for p, _, r, _ in P[s][a])

**常量 `ALPHA`, `BETA`**（来自 `rl_envs.py`）

In [ ]:
ALPHA, BETA = 0.8, 0.7

**常量 `R_SEARCH`, `R_WAIT`, `R_RESCUE`**（来自 `rl_envs.py`）

In [ ]:
R_SEARCH, R_WAIT, R_RESCUE = 2.0, 1.0, -3.0

**函数 `robot_mdp`**（来自 `rl_envs.py`）：返回 P[s][a]；状态 0 = 高电、1 = 低电；动作 0 = 搜索、1 = 等待、2 = 充电（只有低电能充电）。

In [ ]:
def robot_mdp():
    """返回 P[s][a]；状态 0 = 高电、1 = 低电；动作 0 = 搜索、1 = 等待、2 = 充电（只有低电能充电）。"""
    hi, lo = 0, 1
    P = {
        hi: {
            0: [(ALPHA, hi, R_SEARCH, False), (1 - ALPHA, lo, R_SEARCH, False)],
            1: [(1.0, hi, R_WAIT, False)],
        },
        lo: {
            0: [(BETA, lo, R_SEARCH, False), (1 - BETA, hi, R_RESCUE, False)],
            1: [(1.0, lo, R_WAIT, False)],
            2: [(1.0, hi, 0.0, False)],
        },
    }
    return P

**函数 `transition_prob`**（来自 `ch05_mdp.py`）：下面要用到的小工具。

In [ ]:
def transition_prob(P, s, a, s_next):
    return sum(p for p, ns, _, _ in P[s][a] if ns == s_next)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
P = robot_mdp()

In [ ]:
for s in P:
        for a in P[s]:
            rows = "；".join(f"→{ROBOT_STATES[ns]} 奖励 {r:+g} 概率 {p:g}" for p, ns, r, _ in P[s][a])
            print(f"{ROBOT_STATES[s]} + {ROBOT_ACTIONS[a]}：{rows}")

In [ ]:
print("每组概率之和：", set(round(v, 9) for v in check_sums(P).values()))

In [ ]:
print("p(高电 | 低电, 搜索) =", transition_prob(P, 1, 0, 0))

In [ ]:
print("r(低电, 搜索) =", round(expected_reward(P, 1, 0), 4), "；r(高电, 搜索) =", expected_reward(P, 0, 0))

## 3. 代码里才看得到的

1. `P[s][a]` 这种"字典套列表"的写法，和 Gymnasium 里 FrozenLake、Taxi 等环境的 `env.unwrapped.P` 完全一样，以后可以直接拿来跑第 10–12 章的算法；
2. `1 − 0.7` 在电脑里算出来是 `0.30000000000000004`：小数在电脑里存不精确，比较概率时要用"差不多相等"（测试里的 `pytest.approx`），别用 `==`。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 把"低电搜索没电被救"的 −3 改成 −10（复制 `P` 再改 `P[1][0]`），`expected_reward(P, 1, 0)` 从 0.5 变成 −1.6：一步平均奖励就是"概率 × 奖励"加起来。`check_sums` 仍然全是 1.0：改奖励不影响概率。
2. 把 `rl_envs.py` 里的 `ALPHA` 改成 0.5，再看 `check_sums(P)`：两条结果的概率仍然加起来是 1。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 环境是张表：哪个状态做哪个动作，以多大概率到哪、拿多少；
> 竖线右边已知，左边要猜，每组加起来等于一；
> 马尔可夫：现在说明一切，不翻历史；
> 加掉奖励得转移，平均奖励得 r(s, a)。